# Calculo del Índice de Fisher para la Energía


In [79]:
import openpyxl as px
import pandas as pd
import numpy as np

In [80]:
# %% Cargar datos y fijar TIME como datetime, base 2015
import pandas as pd

BASE_YEAR = 2015

data = pd.read_excel('cantidad_depurada_pretax.xlsx').copy()

# 1) Asegurar que TIME son años válidos
time_year = pd.to_numeric(data['TIME'], errors='coerce')
if time_year.isna().any():
    bad_rows = data.loc[time_year.isna(), 'TIME'].head(5).tolist()
    raise ValueError(f"TIME contiene valores no numéricos (ejemplos: {bad_rows})")

# 2) Convertir TIME → datetime (1 enero de cada año)
data['TIME'] = pd.to_datetime(time_year.astype(int).astype(str) + '-01-01')

# 3) Ordenar por fecha
data = data.sort_values('TIME').reset_index(drop=True)

# 4) Seleccionar periodo base
mask_base = data['TIME'].dt.year.eq(BASE_YEAR)
if not mask_base.any():
    years_present = sorted(data['TIME'].dt.year.unique().tolist())
    raise ValueError(f"No hay datos en {BASE_YEAR}. Años disponibles: {years_present}")

base_timestamp = data.loc[mask_base, 'TIME'].min()
print(f"Periodo base fijado en {base_timestamp} (BASE_YEAR={BASE_YEAR})")







Periodo base fijado en 2015-01-01 00:00:00 (BASE_YEAR=2015)


In [81]:
# %% Construir panel largo (t, i, j, P, Q)
import numpy as np
import pandas as pd

Q_cols = {
    ('PETRO','IND'): 'PETRO_IND',
    ('GAS','IND'):   'GAS_IND',
    ('ELEC','IND'):  'ELEC_IND',
    ('PETRO','HOG'): 'PETRO_HOG',
    ('GAS','HOG'):   'GAS_HOG',
    ('ELEC','HOG'):  'ELEC_HOG',
}
P_cols = {
    ('PETRO','IND'): 'PETRO_IND_PRECIO',
    ('GAS','IND'):   'GAS_IND_PRECIO',
    ('ELEC','IND'):  'ELEC_IND_PRECIO',
    ('PETRO','HOG'): 'PETRO_HOG_PRECIO',
    ('GAS','HOG'):   'GAS_HOG_PRECIO',
    ('ELEC','HOG'):  'ELEC_HOG_PRECIO',
}

rows_Q, rows_P = [], []

# Construcción largo Q
for (i, j), col in Q_cols.items():
    tmp = data[['TIME', col]].rename(columns={col: 'Q'})
    tmp['i'], tmp['j'] = i, j
    rows_Q.append(tmp)

# Construcción largo P
for (i, j), col in P_cols.items():
    tmp = data[['TIME', col]].rename(columns={col: 'P'})
    tmp['i'], tmp['j'] = i, j
    rows_P.append(tmp)

# Concatenar y combinar P y Q
panel_Q = pd.concat(rows_Q, ignore_index=True)
panel_P = pd.concat(rows_P, ignore_index=True)

panel_full = (
    panel_Q
    .merge(panel_P, on=['TIME', 'i', 'j'], how='inner')
    .rename(columns={'TIME': 't'})
    .sort_values(['t', 'i', 'j'])
    .reset_index(drop=True)
)

panel_full.head()






,t,Q,i,j,P
0,1978-01-01,2.012812e+06,ELEC,HOG,3.000320e+08
1,1978-01-01,4.164476e+06,ELEC,IND,1.318605e+08
2,1978-01-01,2.120506e+05,GAS,HOG,1.151223e+08
3,1978-01-01,1.132538e+06,GAS,IND,6.611120e+07
4,1978-01-01,1.339745e+06,PETRO,HOG,7.042857e+07


In [82]:
# %% Fisher one-step ratio
import numpy as np
import pandas as pd

def fisher_ratio_one_step(df_long, t, tm1, impute='ffill'):
    """
    Fisher quantity index entre dos periodos adyacentes (tm1 -> t)
    usando df_long con columnas [t, i, j, P, Q].
    """
    df = df_long.copy().sort_values('t')

    # Imputación opcional forward-fill por item (i,j)
    if impute == 'ffill':
        df[['P', 'Q']] = df.groupby(['i', 'j'])[['P', 'Q']].ffill()

    # Filtrar solo para tm1 y t
    sub = df[df['t'].isin([tm1, t])]
    if sub['t'].nunique() < 2:
        return np.nan

    # Pasamos a formato ancho con multi-columnas
    wide = sub.set_index(['i', 'j', 't']).unstack('t')

    # Asegurarnos de que existen ambas columnas de tiempo
    try:
        P_tm1 = wide['P'][tm1]
        P_t   = wide['P'][t]
        Q_tm1 = wide['Q'][tm1]
        Q_t   = wide['Q'][t]
    except KeyError:
        return np.nan

    # Eliminar observaciones con NaN
    mask = P_tm1.notna() & P_t.notna() & Q_tm1.notna() & Q_t.notna()
    if not mask.any():
        return np.nan

    P_tm1, P_t, Q_tm1, Q_t = P_tm1[mask], P_t[mask], Q_tm1[mask], Q_t[mask]

    # Índice de cantidad de Laspeyres (precios del periodo tm1)
    num_L = (P_tm1 * Q_t).sum()
    den_L = (P_tm1 * Q_tm1).sum()

    # Índice de cantidad de Paasche (precios del periodo t)
    num_Pa = (P_t * Q_t).sum()
    den_Pa = (P_t * Q_tm1).sum()

    if den_L == 0 or den_Pa == 0:
        return np.nan

    L = num_L / den_L
    Pa = num_Pa / den_Pa

    if L <= 0 or Pa <= 0:
        return np.nan

    # Índice de Fisher = media geométrica de Laspeyres y Paasche
    return float(np.sqrt(L * Pa))




In [83]:
# %% compute_chain_indices: índices encadenados
def compute_chain_indices(df_long, base_t, impute='ffill'):
    df = df_long.copy()
    ts = sorted(df['t'].unique())

    if base_t not in ts:
        raise ValueError("El periodo base no existe en df_long.")

    # Gasto nominal por periodo
    nom_by_t = (
        df.assign(v=df['P'] * df['Q'])
          .groupby('t')['v']
          .sum()
          .reindex(ts)
    )

    # Cálculo de índices encadenados de cantidad (Q_t)
    Q_level = {base_t: 1.0}   # nivel
    Q_ratio = {base_t: np.nan}  # ratio Fisher entre años consecutivos

    b_idx = ts.index(base_t)

    # Hacia delante
    for k in range(b_idx + 1, len(ts)):
        t, tm1 = ts[k], ts[k - 1]
        r = fisher_ratio_one_step(df, t, tm1, impute=impute)
        Q_ratio[t] = r
        Q_level[t] = Q_level[tm1] * r if (r is not None and not np.isnan(r)) else np.nan

    # Hacia atrás
    for k in range(b_idx - 1, -1, -1):
        t, tp1 = ts[k], ts[k + 1]
        r_fwd = fisher_ratio_one_step(df, tp1, t, impute=impute)
        # r_fwd es el índice de cantidad de t -> tp1
        if r_fwd is None or np.isnan(r_fwd):
            Q_level[t] = np.nan
        else:
            Q_level[t] = Q_level[tp1] / r_fwd

    base_nom = nom_by_t.loc[base_t]

    # "Cantidad equivalente" E_t anotada a precios del año base
    E_t = {t: Q_level[t] * base_nom if not np.isnan(Q_level[t]) else np.nan for t in ts}

    # Índice de precios P_t = gasto nominal / gasto a cantidades constantes
    P_t = {}
    for t in ts:
        if np.isnan(E_t[t]) or E_t[t] == 0:
            P_t[t] = np.nan
        else:
            P_t[t] = nom_by_t.loc[t] / E_t[t]

    out = pd.DataFrame({
        't': ts,
        'Q_ratio': [Q_ratio.get(t, np.nan) for t in ts],
        'Q_t': [Q_level.get(t, np.nan) for t in ts],
        'nominal': [nom_by_t.loc[t] for t in ts],
        'E_t': [E_t[t] for t in ts],
        'P_t': [P_t[t] for t in ts],
    })

    return out




In [84]:
# %% Subíndices por uso (HOG / IND), base 100 en 2015
subs_use = {}

for jj in sorted(panel_full['j'].dropna().unique()):
    df_j = panel_full[panel_full['j'].eq(jj)]
    out_j = compute_chain_indices(df_j, base_t=base_timestamp, impute='ffill')
    out_j = out_j.sort_values('t').reset_index(drop=True)

    # Base 2015 = 100
    Q_base = out_j.loc[out_j['t'].eq(base_timestamp), 'Q_t'].iloc[0]
    P_base = out_j.loc[out_j['t'].eq(base_timestamp), 'P_t'].iloc[0]
    E_base = out_j.loc[out_j['t'].eq(base_timestamp), 'E_t'].iloc[0]

    out_j['Q_t_100']   = (out_j['Q_t'] / Q_base) * 100
    out_j['P_idx_100'] = (out_j['P_t'] / P_base) * 100
    out_j['E_idx_100'] = (out_j['E_t'] / E_base) * 100

    subs_use[jj] = out_j

# Ejemplos
subs_use['HOG'].head(10), subs_use['IND'].head(10)



(           t  Q_ratio       Q_t       nominal           E_t       P_t  \
 0 1978-01-01      NaN  0.340006  7.226762e+14  6.674201e+15  0.108279   
 1 1979-01-01      NaN  0.341424  7.555259e+14  6.702032e+15  0.112731   
 2 1980-01-01      NaN  0.342447  1.054326e+15  6.722129e+15  0.156844   
 3 1981-01-01      NaN  0.338627  1.478273e+15  6.647133e+15  0.222393   
 4 1982-01-01      NaN  0.351362  1.684501e+15  6.897121e+15  0.244232   
 5 1983-01-01      NaN  0.369494  2.302450e+15  7.253049e+15  0.317446   
 6 1984-01-01      NaN  0.376336  2.556939e+15  7.387346e+15  0.346124   
 7 1985-01-01      NaN  0.380859  2.825241e+15  7.476146e+15  0.377901   
 8 1986-01-01      NaN  0.393528  2.952039e+15  7.724819e+15  0.382150   
 9 1987-01-01      NaN  0.451013  3.313050e+15  8.853230e+15  0.374219   
 
      Q_t_100  P_idx_100  E_idx_100  
 0  34.000568  10.827906  34.000568  
 1  34.142352  11.273086  34.142352  
 2  34.244728  15.684404  34.244728  
 3  33.862676  22.239255  33.862

In [85]:
# %% Exportación CSV adaptada a configuración española
import os

os.makedirs('resultados_indices', exist_ok=True)

# ===========================================================
# === 0) Índices totales agregados (toda energía × todo uso)
# ===========================================================
out_total = compute_chain_indices(panel_full, base_t=base_timestamp, impute='ffill')
out_total = out_total.sort_values('t').reset_index(drop=True)

# Base 2015 = 100
Qb = out_total.loc[out_total['t'].eq(base_timestamp), 'Q_t'].iloc[0]
Pb = out_total.loc[out_total['t'].eq(base_timestamp), 'P_t'].iloc[0]
Eb = out_total.loc[out_total['t'].eq(base_timestamp), 'E_t'].iloc[0]

out_total['Q_t_100']   = (out_total['Q_t'] / Qb) * 100
out_total['P_idx_100'] = (out_total['P_t'] / Pb) * 100
out_total['E_idx_100'] = (out_total['E_t'] / Eb) * 100


def exportar_csv_es(df, nombre):
    """
    Exporta DataFrame en formato CSV con:
      - delimitador ';'
      - separador decimal ','
      - codificación 'utf-8-sig'
    """
    ruta = f"resultados_indices/{nombre}.csv"
    df.to_csv(
        ruta,
        sep=';',              # separador de columnas
        decimal=',',          # separador decimal
        encoding='utf-8-sig', # compatible con Excel
        index=False
    )
    print(f"✅ Exportado: {ruta}")


# ===========================================================
# === 1) Total agregado
# ===========================================================
exportar_csv_es(out_total, "indices_total_fisher_pretax_base2015")


# ===========================================================
# === 2) Subíndices por uso (HOG / IND)
# ===========================================================
for jj, df in subs_use.items():
    exportar_csv_es(df, f"indices_use_{jj}_fisher_base2015")


# ===========================================================
# === 3) Subíndices por energía (PETRO / GAS / ELEC)
#     -> primero los construimos si no existen
# ===========================================================
if 'subs_energy' not in globals():

    print("ℹ️ Construyendo subs_energy...")

    subs_energy = {}

    for ii in sorted(panel_full['i'].dropna().unique()):
        df_i = panel_full[panel_full['i'].eq(ii)]
        out_i = compute_chain_indices(df_i, base_t=base_timestamp, impute='ffill')
        out_i = out_i.sort_values('t').reset_index(drop=True)

        # base 2015 = 100
        Qb_i = out_i.loc[out_i['t'].eq(base_timestamp), 'Q_t'].iloc[0]
        Pb_i = out_i.loc[out_i['t'].eq(base_timestamp), 'P_t'].iloc[0]
        Eb_i = out_i.loc[out_i['t'].eq(base_timestamp), 'E_t'].iloc[0]

        out_i['Q_t_100']   = (out_i['Q_t'] / Qb_i) * 100
        out_i['P_idx_100'] = (out_i['P_t'] / Pb_i) * 100
        out_i['E_idx_100'] = (out_i['E_t'] / Eb_i) * 100

        subs_energy[ii] = out_i

# Exportación subíndices por energía
for ii, df in subs_energy.items():
    exportar_csv_es(df, f"indices_energy_{ii}_fisher__pretax_base2015")







✅ Exportado: resultados_indices/indices_total_fisher_pretax_base2015.csv
✅ Exportado: resultados_indices/indices_use_HOG_fisher_base2015.csv
✅ Exportado: resultados_indices/indices_use_IND_fisher_base2015.csv
✅ Exportado: resultados_indices/indices_energy_ELEC_fisher__pretax_base2015.csv
✅ Exportado: resultados_indices/indices_energy_GAS_fisher__pretax_base2015.csv
✅ Exportado: resultados_indices/indices_energy_PETRO_fisher__pretax_base2015.csv
